# 차트, 그래프, 슬라이드 덱 다루기
Claude는 차트, 그래프, 나아가 슬라이드 덱을 다루는 데 매우 뛰어납니다. 사용 사례에 따라 활용해 볼 만한 팁과 요령이 여럿 있습니다. 이 레시피에서는 이런 자료에 Claude를 사용하는 흔한 패턴을 보여 줍니다.

## 차트와 그래프
대체로 Claude와 차트·그래프를 함께 쓰는 일은 간단합니다. 이를 Claude에 넣고 전달하는 방법과, 결과를 개선하는 몇 가지 팁을 살펴보겠습니다.

### 자료 넣기와 Claude API 호출
차트와 그래프를 Claude에 전달하는 가장 좋은 방법은 비전 기능과 PDF 지원 기능을 활용하는 것입니다. 즉 차트나 그래프가 담긴 PDF 문서를 그것에 대한 질문과 함께 Claude에 주는 것입니다.

현재 PDF 기능은 `claude-sonnet-4-6`만 지원합니다. 아직 베타 단계이므로 `pdfs-2024-09-25` 베타 헤더를 함께 제공해야 합니다.

In [ ]:
# Install and create the Anthropic client.
%pip install anthropic

In [2]:
import base64

from anthropic import Anthropic

# While PDF support is in beta, you must pass in the correct beta header
client = Anthropic(default_headers={"anthropic-beta": "pdfs-2024-09-25"})
# For now, only claude-sonnet-4-6 supports PDFs
MODEL_NAME = "claude-sonnet-4-6"

In [37]:
# Make a useful helper function.
def get_completion(messages):
    response = client.messages.create(
        model=MODEL_NAME, max_tokens=8192, temperature=0, messages=messages
    )
    return response.content[0].text

In [12]:
# To start, we'll need a PDF. We will be using the .pdf document located at cvna_2021_annual_report.pdf.
# Start by reading in the PDF and encoding it as base64.
with open("./documents/cvna_2021_annual_report.pdf", "rb") as pdf_file:
    binary_data = pdf_file.read()
    base_64_encoded_data = base64.b64encode(binary_data)
    base64_string = base_64_encoded_data.decode("utf-8")

이 문서를 간단한 질문과 함께 모델에 전달하는 방법을 살펴보겠습니다.

In [13]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64_string,
                },
            },
            {"type": "text", "text": "What's in this document? Answer in a single sentence."},
        ],
    }
]

print(get_completion(messages))

This is a page from Carvana's 2021 Annual Report showing four key metrics: retail units sold, total revenue, total markets at year end, and car vending machines, all displaying significant growth from 2014 to 2021.


꽤 괜찮습니다! 이번에는 좀 더 쓸모 있는 질문을 던져 보겠습니다.

In [15]:
questions = [
    "What was CVNA revenue in 2020?",
    "How many additional markets has Carvana added since 2014?",
    "What was 2016 revenue per retail unit sold?",
]

for index, question in enumerate(questions):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": base64_string,
                    },
                },
                {"type": "text", "text": question},
            ],
        }
    ]

    print(f"\n----------Question {index + 1}----------")
    print(get_completion(messages))


----------Question 1----------
According to the graph showing Total Revenue ($M), Carvana's revenue in 2020 was $5,587 million (or approximately $5.59 billion).

----------Question 2----------
According to the "TOTAL MARKETS AT YEAR END" graph, Carvana started with 4 markets in 2014 and grew to 311 markets by 2021. Therefore, Carvana added 307 markets since 2014 (311 - 4 = 307 additional markets).

----------Question 3----------
Let me calculate this for you:

2016 Revenue: $365 million
2016 Retail Units Sold: 18,761 units

$365 million ÷ 18,761 units = $19,455 per unit (rounded to nearest dollar)

So in 2016, Carvana's revenue per retail unit sold was approximately $19,455.


보시다시피 Claude는 차트와 그래프에 대해 상당히 세밀한 질문에도 답할 수 있습니다. 다만 최대한 활용하기 위한 몇 가지 팁이 있습니다.
- 때로는 Claude의 산술 능력이 발목을 잡습니다. 위 세 번째 질문을 여러 번 실행해 보면 산술을 틀려 최종 답이 어긋나는 경우가 가끔 있습니다. 이런 실수를 막으려면 Claude에 계산기 도구를 주는 것을 고려해 보세요.
- 아주 복잡한 차트와 그래프라면 "먼저 문서에서 보이는 모든 데이터 포인트를 설명하라"고 요청해, 전통적인 생각의 사슬과 비슷한 개선 효과를 이끌어 낼 수 있습니다.
- 그룹이 많은 묶은 막대그래프처럼 색으로 정보를 전달하는 차트에서는 Claude가 어려움을 겪을 때가 있습니다. 그래프의 색을 HEX 코드로 먼저 식별하게 하면 정확도를 높일 수 있습니다.

## 슬라이드 덱
Claude가 차트와 그래프의 달인이라는 것을 알았으니, 차트와 그래프의 진짜 본거지인 슬라이드 덱으로 확장하는 것이 자연스럽습니다!

슬라이드는 금융 서비스를 비롯한 여러 영역에서 중요한 정보 원천입니다. PyPDF 같은 패키지로 슬라이드 덱에서 텍스트를 추출할 *수는* 있지만, 차트와 그래프가 많은 특성 때문에 모델이 정작 필요한 정보에 접근하기 어려워져 좋은 선택이 되지 못하는 경우가 많습니다.

그래서 PDF 지원 기능이 훌륭한 대안이 됩니다. PDF 문서를 처리할 때 추출된 텍스트와 비전을 함께 사용하기 때문입니다. 이 절에서는 Claude에서 PDF 문서로 슬라이드 덱을 검토하는 방법과, 이 방식의 흔한 함정을 다루는 법을 살펴봅니다.

일반적인 슬라이드 덱을 Claude에 넣는 가장 좋은 방법은 PDF로 내려받아 그대로 Claude에 주는 것입니다.

In [17]:
# Open the multi-page PDF document the same way we did earlier.
with open("./documents/twilio_q4_2023.pdf", "rb") as pdf_file:
    binary_data = pdf_file.read()
    base_64_encoded_data = base64.b64encode(binary_data)
    base64_string = base_64_encoded_data.decode("utf-8")

In [18]:
# Now let's pass the document directly to Claude. Note that Claude will process both the text and visual elements of the document.
question = "What was Twilio y/y revenue growth for fiscal year 2023?"
content = [
    {
        "type": "document",
        "source": {"type": "base64", "media_type": "application/pdf", "data": base64_string},
    },
    {"type": "text", "text": question},
]

messages = [{"role": "user", "content": content}]

print(get_completion(messages))

According to the financial results shown in the presentation, Twilio's year-over-year revenue growth for fiscal year 2023 was 9%. This can be found in the "Total Company Results Highlights" section, which shows FY 2023 revenue growth of 9%.


이 방식은 시작하기에 훌륭하고, 어떤 사용 사례에서는 최고의 성능을 냅니다. 다만 몇 가지 한계가 있습니다.
- 한 요청에 제공하는 모든 문서를 합쳐 총 100페이지까지만 넣을 수 있습니다(이 한도는 앞으로 늘릴 예정입니다).
- 슬라이드 내용을 RAG의 일부로 쓴다면, 멀티모달 PDF를 임베딩에 넣는 것이 문제를 일으킬 수 있습니다

다행히 Claude의 비전 기능을 활용하면 일반적인 PDF 텍스트 추출보다 훨씬 품질 좋은 슬라이드 덱의 **텍스트 표현**을 얻을 수 있습니다.

가장 좋은 방법은 현재 슬라이드와 그 이전까지의 서술을 함께 주면서, 덱을 처음부터 끝까지 순차적으로 서술하게 하는 것입니다. 어떻게 하는지 살펴보겠습니다.

In [41]:
# Define a prompt for narrating our slide deck. We would adjut this prompt based on the nature of the deck, but keep the structure largely the same.
prompt = """
You are the Twilio CFO, narrating your Q4 2023 earnings presentation.

The entire earnings presentation document is provided to you.
Please narrate this presentation from Twilio's Q4 2023 Earnings as if you were the presenter. Do not talk about any things, especially acronyms, if you are not exactly sure you know what they mean.

Do not leave any details un-narrated as some of your viewers are vision-impaired, so if you don't narrate every number they won't know the number.

Structure your response like this:
<narration>
    <page_narration id=1>
    [Your narration for page 1]
    </page_narration>

    <page_narration id=2>
    [Your narration for page 2]
    </page_narration>

    ... and so on for each page
</narration>

Use excruciating detail for each page, ensuring you describe every visual element and number present. Show the full response in a single message.
"""
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64_string,
                },
            },
            {"type": "text", "text": prompt},
        ],
    }
]

# Now we use our prompt to narrate the entire deck. Note that this may take a few minutes to run (often up to 10).
completion = get_completion(messages)

In [42]:
import re

# Next we'll parse the response from Claude using regex
pattern = r"<narration>(.*?)</narration>"
match = re.search(pattern, completion.strip(), re.DOTALL)
if match:
    narration = match.group(1)
else:
    raise ValueError("No narration available. Likely due to the model response being truncated.")

텍스트 기반 서술을 얻었으니(완벽하지는 않지만 꽤 괜찮습니다), 이 덱을 텍스트 전용 워크플로 어디에나 쓸 수 있게 되었습니다. 벡터 검색도 포함해서요!

마지막 점검으로, 서술만 가지고 몇 가지 질문을 던져 보겠습니다!

In [43]:
questions = [
    "What percentage of q4 total revenue was the Segment business line?",
    "Has the rate of growth of quarterly revenue been increasing or decreasing? Give just an answer.",
    "What was acquisition revenue for the year ended december 31, 2023 (including negative revenues)?",
]

for index, question in enumerate(questions):
    prompt = f"""You are an expert financial analyst analyzing a transcript of Twilio's earnings call.
Here is the transcript:
<transcript>
{narration}
</transcript>

Please answer the following question:
<question>
{question}
</question>"""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    print(f"\n----------Question {index + 1}----------")
    print(get_completion(messages))


----------Question 1----------
Let me calculate this:

Segment revenue in Q4 2023: $75 million
Total revenue in Q4 2023: $1,076 million

$75M ÷ $1,076M = 0.0697 or approximately 7%

Therefore, the Segment business line represented approximately 7% of Twilio's total Q4 2023 revenue.

----------Question 2----------
Decreasing. The transcript shows Q4 2023 revenue growth was 5% year-over-year, while for the full year 2023 revenue growth was 9% year-over-year, indicating a slowing growth rate. Additionally, the Q1 2024 guidance projects even lower growth of 2-3% year-over-year, confirming the declining trend.

----------Question 3----------
Let me help calculate the acquisition revenue for 2023.

From the transcript, we can see:
- Total revenue for 2023: $4,154 million
- Organic revenue for 2023: $4,146 million

Therefore, acquisition revenue would be:
Total Revenue - Organic Revenue = $4,154M - $4,146M = $8 million

So the acquisition revenue for the year ended December 31, 2023 was $8 m

좋습니다! 이런 기법들을 갖췄으니, 이제 슬라이드 덱처럼 차트와 그래프가 많은 자료에 모델을 적용할 준비가 되었습니다.